<a href="https://colab.research.google.com/github/joseviniciusfreitastenorio/-Multi-Agent-Article-Generator-Gerador-de-Reda-o-Multi-Agentes/blob/main/DialoGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
 #Instalação das dependências

!pip install -q transformers torch pandas accelerate

In [ ]:
# Importação e carregamento do modelo

import re
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executando em dispositivo: {device}")

# Modelo de chat leve e rápido para o Colab
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

In [11]:
#DataFrame dos pedidos
dados = {
    "numero_pedido": ["1001", "1002", "1003", "1004", "1005"],
    "cliente": ["Ana Silva", "Carlos Souza", "Beatriz Lima", "João Pedro", "Mariana Costa"],
    "status": [
        "Em trânsito (previsão de entrega: amanhã)",
        "Entregue com sucesso",
        "Aguardando confirmação de pagamento",
        "Em separação no centro de distribuição",
        "Cancelado pelo cliente"
    ]
}

df_pedidos = pd.DataFrame(dados)
df_pedidos

,numero_pedido,cliente,status
0,1001,Ana Silva,Em trânsito (previsão de entrega: amanhã)
1,1002,Carlos Souza,Entregue com sucesso
2,1003,Beatriz Lima,Aguardando confirmação de pagamento
3,1004,João Pedro,Em separação no centro de distribuição
4,1005,Mariana Costa,Cancelado pelo cliente


In [ ]:
#Funções de busca e palavras-chave

PALAVRAS_CHAVE_STATUS = {
    "status", "pedido", "rastreio", "rastrear", "encomenda",
    "onde está", "entrega", "chegar", "chegou", "situação"
}

def extrair_numero_pedido(texto: str):
    match = re.search(r"\b\d{4,6}\b", texto)
    return match.group(0) if match else None

def verificar_status_pedido(numero_pedido: str, df: pd.DataFrame) -> str:
    resultado = df[df["numero_pedido"] == str(numero_pedido)]
    if not resultado.empty:
        cliente = resultado.iloc[0]["cliente"]
        status = resultado.iloc[0]["status"]
        return f"Pedido #{numero_pedido} (Cliente: {cliente}): {status}."
    return f"Não encontramos nenhum pedido com o número #{numero_pedido}."

In [13]:
#Geração de resposta com o modelo

def gerar_resposta_llm(mensagem_usuario: str, contexto_extra: str = "") -> str:
    system_prompt = (
        "Você é um assistente virtual prestativo de atendimento ao cliente para um e-commerce. "
        "Responda sempre em português, de forma clara, educada e direta."
    )

    conteudo_usuario = mensagem_usuario
    if contexto_extra:
        conteudo_usuario += f"\n\n[Informação do sistema]: {contexto_extra}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": conteudo_usuario}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.3,
            top_p=0.9
        )

    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
#Interface de chat interativa no Colab

print("--- Atendimento Iniciado no Colab (digite 'sair' para encerrar) ---")

while True:
    entrada = input("\nVocê: ").strip()

    if entrada.lower() in ["sair", "exit", "quit"]:
        print("Assistente: Até logo! Atendimento encerrado.")
        break

    if not entrada:
        continue

    entrada_lower = entrada.lower()
    tem_intencao = any(p in entrada_lower for p in PALAVRAS_CHAVE_STATUS)
    numero = extrair_numero_pedido(entrada)

    # 1. Se informou um número (mesmo que seja apenas "1001"), consulta o DataFrame primeiro:
    if numero:
        info = verificar_status_pedido(numero, df_pedidos)
        prompt_extra = f"Dado do banco de dados: '{info}'. Responda ao cliente com essa informação sobre a entrega dele."
        resposta = gerar_resposta_llm(entrada, contexto_extra=prompt_extra)

    # 2. Se falou sobre pedido/entrega, mas não digitou o número:
    elif tem_intencao:
        resposta = "Olá! Por favor, informe o número do seu pedido (ex: 1001) para que eu possa localizar a entrega."

    # 3. Qualquer outra conversa genérica:
    else:
        resposta = gerar_resposta_llm(entrada)

    print(f"Assistente: {resposta}")

--- Atendimento Iniciado no Colab (digite 'sair' para encerrar) ---


In [ ]:
#Instalar dependências (incluindo Gradio)

!pip install -q gradio transformers torch pandas accelerate

In [ ]:
#Gerar o arquivo requirements.txt automaticamente
%%writefile requirements.txt
gradio>=4.0.0
transformers
torch
pandas
accelerate

In [ ]:
#Gerar o arquivo app.py

%%writefile app.py
import re
import torch
import pandas as pd
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto" if device == "cuda" else None
)

if device == "cpu":
    model.to("cpu")

dados = {
    "numero_pedido": ["1001", "1002", "1003", "1004", "1005"],
    "cliente": ["Ana Silva", "Carlos Souza", "Beatriz Lima", "João Pedro", "Mariana Costa"],
    "status": [
        "Em trânsito (previsão de entrega: amanhã)",
        "Entregue com sucesso",
        "Aguardando confirmação de pagamento",
        "Em separação no centro de distribuição",
        "Cancelado pelo cliente"
    ]
}
df_pedidos = pd.DataFrame(dados)

PALAVRAS_CHAVE_STATUS = {
    "status", "pedido", "rastreio", "rastrear", "encomenda",
    "onde está", "entrega", "chegar", "chegou", "situação"
}

def extrair_numero_pedido(texto: str):
    match = re.search(r"\b\d{4,6}\b", texto)
    return match.group(0) if match else None

def verificar_status_pedido(numero_pedido: str, df: pd.DataFrame) -> str:
    resultado = df[df["numero_pedido"] == str(numero_pedido)]
    if not resultado.empty:
        cliente = resultado.iloc[0]["cliente"]
        status = resultado.iloc[0]["status"]
        return f"Pedido #{numero_pedido} (Cliente: {cliente}): {status}."
    return f"Não encontramos nenhum pedido com o número #{numero_pedido}."

def gerar_resposta_llm(mensagem_usuario: str, contexto_extra: str = "") -> str:
    system_prompt = (
        "Você é um assistente virtual prestativo de atendimento ao cliente para um e-commerce. "
        "Responda sempre em português, de forma clara, educada e direta."
    )

    conteudo_usuario = mensagem_usuario
    if contexto_extra:
        conteudo_usuario += f"\n\n[Informação do sistema]: {contexto_extra}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": conteudo_usuario}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.3,
            top_p=0.9
        )

    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def responder(mensagem: str, historico: list):
    mensagem_clean = mensagem.strip()
    if not mensagem_clean:
        return "Por favor, digite uma mensagem válida."

    mensagem_lower = mensagem_clean.lower()
    tem_intencao = any(p in mensagem_lower for p in PALAVRAS_CHAVE_STATUS)
    numero = extrair_numero_pedido(mensagem_clean)

    if numero:
        info = verificar_status_pedido(numero, df_pedidos)
        prompt_extra = f"Dado do banco: '{info}'. Responda ao cliente com essa informação."
        resposta = gerar_resposta_llm(mensagem_clean, contexto_extra=prompt_extra)
    elif tem_intencao:
        resposta = "Olá! Para consultar seu pedido, por favor informe o número dele (ex: 1001)."
    else:
        resposta = gerar_resposta_llm(mensagem_clean)

    return resposta

demo = gr.ChatInterface(
    fn=responder,
    title="Central de Atendimento & Rastreio de Pedidos",
    description="Consulte o status do seu pedido (ex: 1001, 1002...) ou tire dúvidas gerais.",
    examples=["Onde está o meu pedido?", "1001", "Qual a situação do pedido 1003?", "Vocês aceitam Pix?"]
)

if __name__ == "__main__":
    demo.launch(share=True)

In [8]:
# Testar Interface

!python app.py

config.json: 100% 659/659 [00:00<00:00, 2.45MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 17.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 49.4MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 89.9MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 104MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   7% 64.4M/988M [00:00<00:07, 119MB/s, 3.87MB/s  ] 
model.safetensors: reconstructing file:   2% 16.8M/988M [00:00<00:46, 20.9MB/s]
model.safetensors: downloading bytes:  50% 498M/988M [00:02<00:03, 157MB/s, 41.8MB/s  ]
model.safetensors: downloading bytes:  55% 547M/988M [00:03<00:02, 163MB/s, 44.2MB/s  ]
model.safetensors: downloading bytes:  83% 822M/988M [00:04<00:01, 131MB/s, 62.1MB/s  ]
model.safetensors: downloading bytes:  85% 844M/988M [00:05<00:02, 56.5MB/s, 58.7MB/s  ]
model.safetensors: reconstructing file:  81% 805M/988M [00:09<00:02, 74.2MB/s, 40.3MB/s  ]
model.safetensors: downloading bytes: 100% 8